In [1]:
import numpy as np
import pandas as pd
import librosa
import tensorflow as tf
from tensorflow.keras import layers
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report


In [2]:
df_train = pd.read_csv(r"D:\SER_Cross\data\processed\train.csv")
df_val   = pd.read_csv(r"D:\SER_Cross\data\processed\val.csv")
df_test  = pd.read_csv(r"D:\SER_Cross\data\processed\test.csv")

df_all = pd.concat([df_train, df_val, df_test], ignore_index=True)

emotion_map = {
    "neutral": "neutral_calm",
    "calm": "neutral_calm",
    "happy": "happy",
    "sad": "sad",
    "angry": "angry"
}

df_all["emotion_4"] = df_all["emotion"].map(emotion_map)
df_all = df_all.dropna(subset=["emotion_4"])


In [3]:
def extract_logmel(path):
    y, sr = librosa.load(path, sr=22050, duration=4)

    mel = librosa.feature.melspectrogram(
        y=y, sr=sr, n_mels=128,
        n_fft=1024, hop_length=512
    )

    logmel = librosa.power_to_db(mel)
    logmel = (logmel - logmel.mean()) / (logmel.std() + 1e-6)

    if logmel.shape[1] < 173:
        logmel = np.pad(logmel, ((0,0),(0,173-logmel.shape[1])))
    else:
        logmel = logmel[:, :173]

    return logmel


In [4]:
X = np.array([extract_logmel(p) for p in df_all["path"]])[..., np.newaxis]

le = LabelEncoder()
y = le.fit_transform(df_all["emotion_4"])


c:\Users\Asus\anaconda3\Lib\site-packages\paramiko\pkey.py:82: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "cipher": algorithms.TripleDES,
c:\Users\Asus\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.Blowfish and will be removed from this module in 45.0.0.
  "class": algorithms.Blowfish,
c:\Users\Asus\anaconda3\Lib\site-packages\paramiko\transport.py:243: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "class": algorithms.TripleDES,


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)


In [6]:
cnn = tf.keras.Sequential([
    layers.Conv2D(32, (3,3), activation="relu", input_shape=(128,173,1)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3), activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(4, activation="softmax")
])

cnn.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


c:\Users\Asus\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [7]:
cnn.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.1
)


Epoch 1/20
122/122 ━━━━━━━━━━━━━━━━━━━━ 38s 299ms/step - accuracy: 0.2781 - loss: 6.2584 - val_accuracy: 0.3025 - val_loss: 12.1559
Epoch 2/20
122/122 ━━━━━━━━━━━━━━━━━━━━ 37s 301ms/step - accuracy: 0.2923 - loss: 1.3983 - val_accuracy: 0.2194 - val_loss: 5.1700
Epoch 3/20
122/122 ━━━━━━━━━━━━━━━━━━━━ 37s 302ms/step - accuracy: 0.2653 - loss: 1.4122 - val_accuracy: 0.2309 - val_loss: 1.5725
Epoch 4/20
122/122 ━━━━━━━━━━━━━━━━━━━━ 36s 292ms/step - accuracy: 0.2560 - loss: 1.3916 - val_accuracy: 0.2564 - val_loss: 1.4085
Epoch 5/20
122/122 ━━━━━━━━━━━━━━━━━━━━ 35s 288ms/step - accuracy: 0.2542 - loss: 1.3857 - val_accuracy: 0.2517 - val_loss: 1.3922
Epoch 6/20
122/122 ━━━━━━━━━━━━━━━━━━━━ 36s 294ms/step - accuracy: 0.2555 - loss: 1.3851 - val_accuracy: 0.2540 - val_loss: 1.3821
Epoch 7/20
122/122 ━━━━━━━━━━━━━━━━━━━━ 35s 290ms/step - accuracy: 0.2553 - loss: 1.3918 - val_accuracy: 0.2517 - val_loss: 1.3909
Epoch 8/20
122/122 ━━━━━━━━━━━━━━━━━━━━ 36s 295ms/step - accuracy: 0.2548 - loss: 

In [8]:
crnn = tf.keras.models.load_model(r"D:\SER_Cross\models\crnn_4class_model.h5")


In [12]:
X_test_crnn = np.transpose(X_test.squeeze(-1), (0, 2, 1))
crnn_probs = crnn.predict(X_test_crnn)


46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step


In [13]:
ensemble_probs = 0.7 * crnn_probs + 0.3 * cnn_probs
y_pred = np.argmax(ensemble_probs, axis=1)


In [14]:
acc = accuracy_score(y_test, y_pred)
print("Ensemble Test Accuracy:", acc)


Ensemble Test Accuracy: 0.6773074253990284
